In [ ]:
#import tensorflow and check version
import tensorflow as tf
print(tf.__version__)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

def plot_series(time, series, format='-', start=0, end=None):
    plt.plot(time[start:end], series[start:end], format)
    plt.xlabel('Time')
    plt.ylabel('Value')
    plt.grid(True)

In [ ]:
import csv
time_step = []
temps = []

### read in the data
with open('CPI_PPI_2001_2022.csv') as csvfile:
    reader = csv.reader(csvfile, delimiter = ',')
    next(reader)
    step = 0
    for row in reader:
        temps.append(float(row[1]))
        time_step.append(step)
        step = step + 1
        
### remember to remove null price entry points in history
series = np.array(temps)
series1 = np.array(temps)

time = np.array(time_step)
plt.figure(figsize=(10,6), facecolor='white')
plot_series(time, series)

series = series.reshape(-1, 1)

In [ ]:
from sklearn.preprocessing import MinMaxScaler

#Scale all of the data to be values between 0 and 1
scaler = MinMaxScaler(feature_range=(0,1))
series = scaler.fit_transform(series)
plt.figure(figsize=(10,6), facecolor='white')
plot_series(time, series)
print(len(series))

In [ ]:
#Pick around 80% of data for training
split_time = 208
time_train = time[:split_time]
x_train = series[:split_time]
time_valid = time[split_time:]
x_valid = series[split_time:]

window_size = 6 # 6 months
batch_size = 6

plt.figure(figsize=(10,6), facecolor='white')
plot_series(time_train, x_train)
plot_series(time_valid, x_valid)

In [ ]:
def windowed_dataset(series, window_size, batch_size):
    series = tf.expand_dims(series, axis=-1)
    ds = tf.data.Dataset.from_tensor_slices(series)
    ds = ds.window(window_size+1, shift=1, drop_remainder=True)
    ds = ds.flat_map(lambda w: w.batch(window_size+1))
    ds = ds.map(lambda w: (w[:-1], w[1:]))
    return ds.batch(batch_size).prefetch(1)

In [ ]:
def model_forecast(model, series, window_size):
    ds = tf.data.Dataset.from_tensor_slices(series)
    ds = ds.window(window_size, shift=1, drop_remainder=True)
    ds = ds.flat_map(lambda w: w.batch(window_size))
    ds = ds.batch(6).prefetch(1)
    forecast = model.predict(ds)
    return forecast

In [ ]:
tf.keras.backend.clear_session()
tf.random.set_seed(51)
np.random.seed(51)
window_size = 6
batch_size = 6
train_set = windowed_dataset(x_train, window_size, batch_size)
print(x_train.shape)

In [ ]:
model = tf.keras.models.Sequential([
    tf.keras.layers.Conv1D(filters=30, kernel_size=3,
                         strides=1, padding='causal',
                         activation='relu', input_shape=[None, 1]),
    tf.keras.layers.LSTM(30, return_sequences=True),
    tf.keras.layers.LSTM(30, return_sequences=False),
    tf.keras.layers.Dense(30, activation='relu'),
    tf.keras.layers.Dense(10, activation='relu'),
    #tf.keras.layers.Dropout(0,1), #drop out is no good in this case!!!
    tf.keras.layers.Dense(1)
])

In [ ]:
#lr_schedule = tf.keras.callbacks.LearningRateScheduler(lambda epoch: 1e-8 * 10 ** (epoch/20))
#optimizer = tf.keras.optimizers.SGD(lr=1e-8, momentum=0.9)
#model.compile(loss=tf.keras.losses.Huber(), optimizer=optimizers, metrics=['mae'])

model.compile(optimizer='adam', loss= 'mse')

#history = model.fit(train_set, epochs=100, callbacks=[lr_schedule])
history = model.fit(train_set, epochs=100)

In [ ]:
loss = history.history['loss']
epochs = range(len(loss))
plt.plot(epochs, loss, 'r')

In [ ]:
model.summary()

rnn_forecast = model_forecast(model, series[..., np.newaxis], window_size)
rnn_forecast1 = scaler.inverse_transform(rnn_forecast) #undo scaling
rnn_forecast1 = rnn_forecast1[split_time - window_size:-1, :]

In [ ]:
x_train1 = scaler.inverse_transform(x_train)
x_valid1 = scaler.inverse_transform(x_valid)

plt.figure(figsize=(10,6), facecolor='white')
plot_series(time_train, x_train1)
plot_series(time_valid, x_valid1)
plot_series(time_valid, rnn_forecast1)
plt.xlabel('Time')
plt.ylabel('CPI')
plt.title('CPI Growth Rate')
plt.grid(True)

print(x_valid1[1:6])
print()
print(rnn_forecast1[1:6])

In [ ]:
tf.keras.metrics.mean_squared_error(x_valid1, rnn_forecast1).numpy()

print(rnn_forecast1)

In [ ]:
### Let's do some prediction for Jan 4th, 2022!
# Get the last 6 day closing price, data is already scaled with values between 0 and 1
last_6_days = series[-6:]
#Create an empty list
X_test = []
#Append the past 6 days
X_test.append(last_6_days)
#Convert the X_test data set to a numpy array
X_test = np.array(X_test)
#Reshape the data
X_test = np.reshape(X_test, (X_test.shape[0], X_test.shape[1], 1))
#Get the predicted scaled price
pred_price = model.predict(X_test)
#undo the scaling
pred_price = scaler.inverse_transform(pred_price)
print('predicted_price:', pred_price)
print()

In [ ]:
plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] = False

ir_df = pd.read_csv('CPI_PPI_2001_2022.csv')
#date = pd.to_datetime(ir_df['Date'], format='%Y/%m/%d')
date = pd.to_datetime(ir_df['Date'], format='%Y-%m-%d')

time_train = date[:split_time]
time_valid = date[split_time:]

time_valid


In [ ]:
plt.figure(figsize=(10,6), facecolor='white')
plot_series(time_train, x_train1)
plot_series(time_valid, x_valid1)
plot_series(time_valid, rnn_forecast1)
plt.xlabel('时间')
plt.ylabel('CPI消费者价格指数')
plt.title('LSTM: CPI消费者价格指数同比增长率')

plt.savefig('CPI_China_LSTM.png', bbox_inches='tight')
plt.show()